# Himawari-9 ingest + processing (Kaggle, CPU)

Bronze → silver → gold for August 2025 over Ho Chi Minh City, using
[hcmc-storm-nowcasting](https://github.com/ngocquangbiii06/hcmc-storm-nowcasting).

**Settings:** Internet **on**, accelerator **None**. Run with **Save Version → Save & Run All** so it runs in
the background (the PC can be turned off). Output (`/kaggle/working/data`): daily silver NetCDF,
Parquet catalog, gold Zarr and the data-quality report. Raw files are deleted as soon as they are processed.

In [ ]:
!git clone -q --depth 1 https://github.com/ngocquangbiii06/hcmc-storm-nowcasting.git /tmp/stormcast
!pip install -q -e "/tmp/stormcast"
import os, sys, importlib
os.chdir('/tmp/stormcast'); sys.path.insert(0, '/tmp/stormcast/src')
# fail loudly here rather than 141 times per day later
for mod in ['satpy', 'pyresample', 'xarray', 'zarr', 'boto3'] + []:
    importlib.import_module(mod)
print(sys.version.split()[0], 'dependencies ok')

In [ ]:
import time, logging
from dataclasses import replace
import pandas as pd
from stormcast import pipeline
from stormcast.config import load_config
from stormcast.ingest.himawari import wanted_segments

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
START, END = '2025-08-01', '2025-08-31'
TIME_BUDGET_H = 11.0  # Kaggle stops a run at 12 h; leave time for build + report

cfg = load_config('configs/default.yaml')
cfg = replace(
    cfg,
    period=replace(cfg.period, start=START, end=END),
    paths=replace(cfg.paths, data_root='/kaggle/working/data', reports='/kaggle/working/reports'),
    himawari=replace(cfg.himawari, keep_raw=False, download_workers=32),
    processing=replace(cfg.processing, workers=os.cpu_count()),
)
print('segments', wanted_segments(cfg), '| cpus', os.cpu_count())

In [ ]:
t_start, timings = time.time(), []
for day in pd.date_range(START, END).date:
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print('time budget reached, stopping before', day); break
    t0 = time.time(); raw = pipeline.himawari_download(cfg, day)
    t1 = time.time(); frames = pipeline.himawari_process(cfg, day)
    t2 = time.time()
    timings.append({'day': str(day), 'files': len(raw), 'gb': raw['size_bytes'].sum() / 1e9 if len(raw) else 0,
                    'download_s': round(t1 - t0), 'process_s': round(t2 - t1),
                    'frames_ok': int((frames['status'] == 'ok').sum()) if len(frames) else 0,
                    'frames_total': len(frames)})
    print(timings[-1], flush=True)
    pd.DataFrame(timings).to_csv('/kaggle/working/timings.csv', index=False)
    if timings[-1]['frames_total'] and not timings[-1]['frames_ok']:
        raise RuntimeError(f'no frame of {day} processed - see warnings above; stopping instead of wasting the run')

In [ ]:
print(pipeline.build_sat(cfg))
from stormcast.report import build_report
print(open(build_report(cfg)).read())
t = pd.DataFrame(timings)
print('GB downloaded:', round(t.gb.sum(), 1), '| frames ok:', t.frames_ok.sum(), '/', t.frames_total.sum(),
      '| median min/day:', round((t.download_s + t.process_s).median() / 60, 1))
!du -sh /kaggle/working/data/*